In [ ]:
# uv vesion
!uv --version

uv 0.13.4 (7456b60bc 2026-10-20 x86_64-pc-windows-msvc)


In [2]:
# scikit version
import sklearn
print(sklearn.__version__)

1.7.2


In [4]:
# conversion probability (loaded model)
import pickle

with open('pipeline.bin', 'rb') as f:
    model = pickle.load(f)

lead_q3 = {
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}

probability = model.predict_proba([lead_q3])[0, 1]
print(f"Conversion Probability: {probability:.3f}")

Conversion Probability: 0.533


In [5]:
# conversion probability (api)
import requests

url = "http://localhost:9696/predict"
lead_q4 = {
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}

response = requests.post(url, json=lead_q4)
result = response.json()

print(f"Conversion Probability (API): {result.get('conversion_probability', 0):.3f}")

Conversion Probability (API): 0.770


In [6]:
# python image
with open("Dockerfile", "r") as file:
    print("Base image:", file.readline().strip())

Base image: FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


In [8]:
# conversion probability (container)
import time
import requests

!docker build -t zoomcamp-model:2026-hw5 .

!docker run -d --rm -p 9696:9696 --name zoomcamp-hw5 zoomcamp-model:2026-hw5

time.sleep(3)

health = requests.get("http://localhost:9696/health").json()
print("Health Check:", health)

response = requests.post("http://localhost:9696/predict", json=lead_q4)
result = response.json()
print(f"Conversion Probability (Container): {result.get('conversion_probability', 0):.3f}")

!docker stop zoomcamp-hw5

Sending build context to Docker daemon  298.2MB

Step 1/9 : FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
 ---> d29f48a31a8b
Step 2/9 : COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/
 ---> Using cache
 ---> 8368106fb67c
Step 3/9 : ENV PATH="/code/.venv/bin:$PATH"     UV_COMPILE_BYTECODE=1     UV_LINK_MODE=copy
 ---> Using cache
 ---> c4b124516bb4
Step 4/9 : WORKDIR /code
 ---> Using cache
 ---> 585f76c5ec8b
Step 5/9 : COPY pyproject.toml uv.lock .python-version ./
 ---> Using cache
 ---> 95873d3b7bed
Step 6/9 : RUN uv sync --locked --no-dev
 ---> Using cache
 ---> 01a3d0f53651
Step 7/9 : COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./
 ---> Using cache
 ---> dde9074aeab1
Step 8/9 : EXPOSE 9696
 ---> Using cache
 ---> 3a4e266311fa
Step 9/9 : ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

docker: Error response from daemon: Conflict. The container name "/zoomcamp-hw5" is already in use by container "f9014cb2116a9482bf0b185ae0aea281752b295f725b1d175d5626700b3eabfb". You have to remove (or rename) that container to be able to reuse that name.

Run 'docker run --help' for more information


Health Check: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '5cb9195a7a160fdc91a1b8f57172055e33ced1994870742b937d5273f4021c67'}
Conversion Probability (Container): 0.770
zoomcamp-hw5
